In [64]:
from pathlib import Path
import os
import sys
import json
import hashlib
import subprocess
import importlib.util

import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    r"C:\Users\hp\Nassau_Candy_Final_Submission"
)

NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"

APP_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "app_bundle"
)

APP_DATA = APP_ROOT / "data"
APP_MODELS = APP_ROOT / "models"
APP_REPORTS = APP_ROOT / "reports"
APP_SRC = APP_ROOT / "src"

APP_FILE = APP_ROOT / "app.py"

print("Starting project setup...")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project folder not found: {PROJECT_ROOT}"
    )

print("Project folder found.")

folders = [
    NOTEBOOK_DIR,
    APP_ROOT,
    APP_DATA,
    APP_MODELS,
    APP_REPORTS,
    APP_SRC
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)
    print("Folder ready:", folder)

print("\nPROJECT SETUP COMPLETED")
print("Project root:", PROJECT_ROOT)
print("Notebook folder:", NOTEBOOK_DIR)
print("Application folder:", APP_ROOT)
print("Application file:", APP_FILE)
print("Application already exists:", APP_FILE.exists())

Starting project setup...
Project folder found.
Folder ready: C:\Users\hp\Nassau_Candy_Final_Submission\notebooks
Folder ready: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle
Folder ready: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\data
Folder ready: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\models
Folder ready: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\reports
Folder ready: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\src

PROJECT SETUP COMPLETED
Project root: C:\Users\hp\Nassau_Candy_Final_Submission
Notebook folder: C:\Users\hp\Nassau_Candy_Final_Submission\notebooks
Application folder: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle
Application file: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\app.py
Application already exists: True


In [66]:
# Inspect project files and identify the processed dataset

print("INSPECTING APPLICATION BUNDLE")

csv_files = sorted(APP_ROOT.rglob("*.csv"))

model_files = sorted(
    list(APP_ROOT.rglob("*.joblib")) +
    list(APP_ROOT.rglob("*.pkl"))
)

json_files = sorted(APP_ROOT.rglob("*.json"))

python_files = sorted(APP_SRC.glob("*.py"))

print("\nCSV FILES:", len(csv_files))

for file in csv_files:
    print(file.relative_to(APP_ROOT))

print("\nMODEL FILES:", len(model_files))

for file in model_files:
    print(file.relative_to(APP_ROOT))

print("\nJSON FILES:", len(json_files))

for file in json_files:
    print(file.relative_to(APP_ROOT))

print("\nPYTHON SOURCE FILES:", len(python_files))

for file in python_files:
    print(file.name)

print("\nEXISTING APP FILE:")
print(APP_FILE)
print("Exists:", APP_FILE.exists())

INSPECTING APPLICATION BUNDLE

CSV FILES: 46
data\factory_master.csv
data\kpis\high_derived_lead_time_risk.csv
data\kpis\kpi_average_derived_lead_time.csv
data\kpis\kpi_dashboard_data.csv
data\kpis\kpi_estimated_lead_time_reduction.csv
data\kpis\kpi_export_manifest.csv
data\kpis\kpi_gross_profit.csv
data\kpis\kpi_lead_time_variability.csv
data\kpis\kpi_median_derived_lead_time.csv
data\kpis\kpi_profit_margin.csv
data\kpis\kpi_total_sales.csv
data\kpis\low_margin_risk.csv
data\kpis\low_sample_size_risk.csv
data\kpis\model_uncertainty_report.csv
data\kpis\phase11_integrity_report.csv
data\kpis\profit_stability_proxy.csv
data\kpis\recommendation_coverage.csv
data\kpis\region_performance_risk.csv
data\kpis\region_sample_size_risk.csv
data\kpis\risk_register.csv
data\kpis\scenario_support_score.csv
data\processed_dataset.csv
data\product_factory_mapping.csv
reports\application_runtime_report.csv
reports\application_validation_report.csv
reports\complete_application_validation.csv
reports\mo

In [68]:
print("IDENTIFYING PROCESSED TRANSACTION DATASET")

required_columns = {
    "Order Date",
    "Ship Date",
    "Ship Mode",
    "Product Name",
    "Sales",
    "Units",
    "Gross Profit",
    "Cost",
    "Region"
}

dataset_candidates = []

for file in csv_files:

    try:
        sample = pd.read_csv(file, nrows=5)

        if required_columns.issubset(sample.columns):

            dataset_candidates.append(file)

            print("\nMatching dataset found:")
            print(file)

            print("Columns:", len(sample.columns))

    except Exception as error:
        print("Could not inspect:", file.name)
        print(error)

if not dataset_candidates:

    raise FileNotFoundError(
        "Processed transaction dataset was not found."
    )

PROCESSED_DATA_PATH = max(
    dataset_candidates,
    key=lambda file: file.stat().st_size
)

print("\nSELECTED PROCESSED DATASET")
print(PROCESSED_DATA_PATH)

transactions = pd.read_csv(PROCESSED_DATA_PATH)

print("\nDATASET SHAPE:", transactions.shape)

display(transactions.head())

print("\nDATASET IDENTIFICATION COMPLETED")

IDENTIFYING PROCESSED TRANSACTION DATASET

Matching dataset found:
C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\data\processed_dataset.csv
Columns: 34

SELECTED PROCESSED DATASET
C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\data\processed_dataset.csv

DATASET SHAPE: (10194, 34)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Order_Weekday,Ship_Year,Ship_Month,Profit_Margin_Pct,Sales_Per_Unit,Cost_Per_Unit,Profit_Per_Unit,Lead_Time_Category,Profit_Category,Sales_Category
0,1,US-2021-103800-CHO-MIL-31000,03-01-2024,30-06-2026,Standard Class,103800,United States,Houston,Texas,77095,...,Wednesday,2026,6,64.923077,3.25,1.14,2.11,Low,Low,Low
1,2,US-2021-112326-CHO-TRI-54000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,...,Thursday,2026,7,65.333333,3.75,1.30,2.45,Low,Low,Low
2,3,US-2021-112326-CHO-NUT-13000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,...,Thursday,2026,7,71.346705,3.49,1.00,2.49,Low,Medium,Medium
3,4,US-2021-112326-CHO-SCR-58000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,...,Thursday,2026,7,69.444444,3.60,1.10,2.50,Low,Medium,Medium
4,5,US-2021-141817-CHO-TRI-54000,05-01-2024,05-07-2026,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,...,Friday,2026,7,65.333333,3.75,1.30,2.45,Low,Medium,Medium



DATASET IDENTIFICATION COMPLETED


In [70]:
# Validate dataset structure and data types 

print("VALIDATING PROCESSED DATASET")

print("\nDATASET DIMENSIONS")
print("Rows:", transactions.shape[0])
print("Columns:", transactions.shape[1])

print("\nCOLUMN NAMES")

for column in transactions.columns:
    print("-", column)

print("\nDATA TYPES")

display(
    transactions.dtypes.to_frame(
        name="Data Type"
    )
)

print("\nMISSING VALUES")

missing_report = pd.DataFrame({
    "Missing Values": transactions.isnull().sum(),
    "Missing Percentage": (
        transactions.isnull().mean() * 100
    ).round(2)
})

display(missing_report)

print("\nDUPLICATE ROWS")
print(transactions.duplicated().sum())

print("\nNUMERIC SUMMARY")

display(
    transactions[
        transactions.select_dtypes(
            include=np.number
        ).columns
    ].describe().T
)

print("\nDATASET STRUCTURE VALIDATION COMPLETED")

VALIDATING PROCESSED DATASET

DATASET DIMENSIONS
Rows: 10194
Columns: 34

COLUMN NAMES
- Row ID
- Order ID
- Order Date
- Ship Date
- Ship Mode
- Customer ID
- Country/Region
- City
- State/Province
- Postal Code
- Division
- Region
- Product ID
- Product Name
- Sales
- Units
- Gross Profit
- Cost
- Order_Date_Parsed
- Ship_Date_Parsed
- Lead_Time_Days
- Order_Year
- Order_Month
- Order_Quarter
- Order_Weekday
- Ship_Year
- Ship_Month
- Profit_Margin_Pct
- Sales_Per_Unit
- Cost_Per_Unit
- Profit_Per_Unit
- Lead_Time_Category
- Profit_Category
- Sales_Category

DATA TYPES


,Data Type
Row ID,int64
Order ID,object
Order Date,object
Ship Date,object
Ship Mode,object
Customer ID,int64
Country/Region,object
City,object
State/Province,object
Postal Code,object



MISSING VALUES


,Missing Values,Missing Percentage
Row ID,0,0.0
Order ID,0,0.0
Order Date,0,0.0
Ship Date,0,0.0
Ship Mode,0,0.0
Customer ID,0,0.0
Country/Region,0,0.0
City,0,0.0
State/Province,0,0.0
Postal Code,0,0.0



DUPLICATE ROWS
0

NUMERIC SUMMARY


,count,mean,std,min,25%,50%,75%,max
Row ID,10194.0,5097.500000,2942.898656,1.000000,2549.250000,5097.500000,7645.750000,10194.0
Customer ID,10194.0,134468.961154,20231.483007,100006.000000,117212.000000,133550.000000,152051.000000,192314.0
Sales,10194.0,13.908537,11.341020,1.250000,7.200000,10.800000,18.000000,260.0
Units,10194.0,3.791838,2.228317,1.000000,2.000000,3.000000,5.000000,14.0
Gross Profit,10194.0,9.166451,6.643740,0.250000,4.900000,7.470000,12.250000,130.0
Cost,10194.0,4.742087,5.061647,0.600000,2.400000,3.600000,5.700000,130.0
Lead_Time_Days,10194.0,1320.841868,262.444892,904.000000,1271.000000,1274.000000,1638.000000,1642.0
Order_Year,10194.0,2024.589857,0.491884,2024.000000,2024.000000,2025.000000,2025.000000,2025.0
Order_Month,10194.0,7.811850,3.295915,1.000000,5.000000,9.000000,11.000000,12.0
Order_Quarter,10194.0,2.882676,1.060316,1.000000,2.000000,3.000000,4.000000,4.0



DATASET STRUCTURE VALIDATION COMPLETED


In [72]:
# Validate business data and prepare dashboard dataset
print("PREPARING DASHBOARD DATA")

required_columns = [
    "Order ID",
    "Order Date",
    "Ship Date",
    "Ship Mode",
    "Product Name",
    "Region",
    "Sales",
    "Units",
    "Gross Profit",
    "Cost"
]

missing_columns = [
    column
    for column in required_columns
    if column not in transactions.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns missing: {missing_columns}"
    )

dashboard_data = transactions.copy()

dashboard_data["_Order_Date"] = pd.to_datetime(
    dashboard_data["Order Date"],
    dayfirst=True,
    errors="coerce"
)

dashboard_data["_Ship_Date"] = pd.to_datetime(
    dashboard_data["Ship Date"],
    dayfirst=True,
    errors="coerce"
)

dashboard_data["_Lead_Time_Days"] = (
    dashboard_data["_Ship_Date"]
    - dashboard_data["_Order_Date"]
).dt.days

numeric_columns = [
    "Sales",
    "Units",
    "Gross Profit",
    "Cost"
]

for column in numeric_columns:
    dashboard_data[column] = pd.to_numeric(
        dashboard_data[column],
        errors="coerce"
    )

print("Dashboard dataset created.")

print("Rows:", len(dashboard_data))
print("Columns:", len(dashboard_data.columns))

print("\nUnique products:",
      dashboard_data["Product Name"].nunique())

print("Unique regions:",
      dashboard_data["Region"].nunique())

print("Unique ship modes:",
      dashboard_data["Ship Mode"].nunique())

print("Average derived lead time:",
      round(
          dashboard_data["_Lead_Time_Days"].mean(),
          2
      ))

display(dashboard_data.head())

print("\nDASHBOARD DATA PREPARATION COMPLETED")


PREPARING DASHBOARD DATA
Dashboard dataset created.
Rows: 10194
Columns: 37

Unique products: 15
Unique regions: 4
Unique ship modes: 4
Average derived lead time: 1320.84


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Profit_Margin_Pct,Sales_Per_Unit,Cost_Per_Unit,Profit_Per_Unit,Lead_Time_Category,Profit_Category,Sales_Category,_Order_Date,_Ship_Date,_Lead_Time_Days
0,1,US-2021-103800-CHO-MIL-31000,03-01-2024,30-06-2026,Standard Class,103800,United States,Houston,Texas,77095,...,64.923077,3.25,1.14,2.11,Low,Low,Low,2024-01-03,2026-06-30,909
1,2,US-2021-112326-CHO-TRI-54000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,...,65.333333,3.75,1.30,2.45,Low,Low,Low,2024-01-04,2026-07-01,909
2,3,US-2021-112326-CHO-NUT-13000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,...,71.346705,3.49,1.00,2.49,Low,Medium,Medium,2024-01-04,2026-07-01,909
3,4,US-2021-112326-CHO-SCR-58000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,...,69.444444,3.60,1.10,2.50,Low,Medium,Medium,2024-01-04,2026-07-01,909
4,5,US-2021-141817-CHO-TRI-54000,05-01-2024,05-07-2026,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,...,65.333333,3.75,1.30,2.45,Low,Medium,Medium,2024-01-05,2026-07-05,912



DASHBOARD DATA PREPARATION COMPLETED


In [74]:
# Inspect available dashboard and recommendation data
# This cell identifies the existing KPI, recommendation, risk, factory, and model files so the app uses your actual project outputs rather than invented data

print("INSPECTING DASHBOARD SUPPORTING DATA")

keywords = [
    "kpi",
    "recommend",
    "risk",
    "factory",
    "region",
    "model",
    "scenario",
    "product",
    "ship",
    "comparison",
    "metadata",
    "schema"
]

supporting_files = []

for file in csv_files + json_files:
    if any(word in file.name.lower() for word in keywords):
        supporting_files.append(file)

for file in supporting_files:
    print("\nFILE:", file.relative_to(APP_ROOT))

    try:
        if file.suffix.lower() == ".csv":
            sample = pd.read_csv(file, nrows=3)
            print("Shape:", pd.read_csv(file).shape)
            print("Columns:", sample.columns.tolist())
            display(sample.head(2))

        elif file.suffix.lower() == ".json":
            with open(file, "r", encoding="utf-8") as handle:
                content = json.load(handle)

            print("JSON type:", type(content).__name__)

            if isinstance(content, dict):
                print("Keys:", list(content.keys())[:30])

    except Exception as error:
        print("Inspection error:", error)

print("\nSUPPORTING DATA INSPECTION COMPLETED")

INSPECTING DASHBOARD SUPPORTING DATA

FILE: data\factory_master.csv
Shape: (5, 3)
Columns: ['Factory_Name', 'Latitude', 'Longitude']


,Factory_Name,Latitude,Longitude
0,Lot's O' Nuts,32.881893,-111.768036
1,Wicked Choccy's,32.076176,-81.088371



FILE: data\kpis\high_derived_lead_time_risk.csv
Shape: (15, 12)
Columns: ['Product Name', 'Order_Lines', 'Unique_Orders', 'Mean_Lead_Time_Days', 'Median_Lead_Time_Days', 'Lead_Time_Std_Days', 'Lead_Time_IQR_Days', 'Threshold_Days', 'Sample_Eligible', 'High_Lead_Time_Risk_Flag', 'Risk_Basis', 'Limitation']


,Product Name,Order_Lines,Unique_Orders,Mean_Lead_Time_Days,Median_Lead_Time_Days,Lead_Time_Std_Days,Lead_Time_IQR_Days,Threshold_Days,Sample_Eligible,High_Lead_Time_Risk_Flag,Risk_Basis,Limitation
0,Everlasting Gobstopper,3,3,1394.666667,1273.0,213.336198,185.5,1361.119149,False,False,Product mean derived lead-time at or above the...,Descriptive historical flag only. Derived date...
1,Fizzy Lifting Drinks,6,6,1273.500000,1274.0,326.691751,546.0,1361.119149,True,False,Product mean derived lead-time at or above the...,Descriptive historical flag only. Derived date...



FILE: data\kpis\kpi_average_derived_lead_time.csv
Shape: (1, 7)
Columns: ['KPI', 'Value', 'Unit', 'Scope', 'Valid_Record_Count', 'Status', 'Interpretation']


,KPI,Value,Unit,Scope,Valid_Record_Count,Status,Interpretation
0,Average Derived Lead Time,1320.841868,Days,Rows with both dates successfully parsed,10194,Historical date difference,Order Date-to-Ship Date difference only. The u...


... (55 outputs left out)

,Coverage_Metric,Count,Interpretation,Coverage_Percent
0,Source products,15,Products present in the original CSV,100.0
1,Products with reference mapping,15,Products with a Phase 07 reference mapping,100.0



FILE: reports\recommendations\task_10_18_unsupported_scenarios.csv
Shape: (60, 13)
Columns: ['Product Name', 'Reference Factory', 'Factory Option', 'Is_Reference_Mapping', 'Option_Type', 'Compatibility_Status', 'Factory_Specific_Lead_Time_Estimable', 'Factory_Specific_Cost_Available', 'Capacity_Data_Available', 'Warning_Code', 'Warning_Reason', 'Permitted_Use', 'Prohibited_Claim']


,Product Name,Reference Factory,Factory Option,Is_Reference_Mapping,Option_Type,Compatibility_Status,Factory_Specific_Lead_Time_Estimable,Factory_Specific_Cost_Available,Capacity_Data_Available,Warning_Code,Warning_Reason,Permitted_Use,Prohibited_Claim
0,Everlasting Gobstopper,Secret Factory,Lot's O' Nuts,False,Alternative factory - unverified,UNVERIFIED,False,False,False,UNVERIFIED_ALTERNATIVE_FACTORY,"No verified product-factory compatibility, his...",Exploratory screening and data collection only,"Do not claim feasibility, shorter lead time, l..."
1,Everlasting Gobstopper,Secret Factory,Sugar Shack,False,Alternative factory - unverified,UNVERIFIED,False,False,False,UNVERIFIED_ALTERNATIVE_FACTORY,"No verified product-factory compatibility, his...",Exploratory screening and data collection only,"Do not claim feasibility, shorter lead time, l..."



FILE: models\feature_schema.json
JSON type: dict
Keys: ['target_variable', 'predictor_features', 'feature_count', 'selected_model']

FILE: models\model_metadata.json
JSON type: dict
Keys: ['project', 'target_variable', 'selected_model', 'predictor_features', 'training_records', 'testing_records', 'test_MAE', 'test_RMSE', 'test_R2', 'model_file']

FILE: reports\model_metrics\model_metadata.json
JSON type: dict
Keys: ['project', 'target_variable', 'selected_model', 'predictor_features', 'training_records', 'testing_records', 'test_MAE', 'test_RMSE', 'test_R2', 'model_file']

SUPPORTING DATA INSPECTION COMPLETED


In [76]:
# Discover and load supporting CSV files

print("DISCOVERING SUPPORTING DATASETS")

supporting_data = {}
supporting_inventory = []

for file in csv_files:
    try:
        sample = pd.read_csv(file, nrows=5)

        supporting_inventory.append({
            "File": file.name,
            "Path": str(file),
            "Columns": sample.columns.tolist(),
            "Column Count": len(sample.columns)
        })

    except Exception as error:
        print(f"Could not inspect {file.name}: {error}")

supporting_inventory_df = pd.DataFrame(
    supporting_inventory
)

if not supporting_inventory_df.empty:
    display(
        supporting_inventory_df[
            ["File", "Column Count", "Columns"]
        ]
    )

    for item in supporting_inventory:
        file = Path(item["Path"])

        try:
            supporting_data[file.stem] = pd.read_csv(file)
        except Exception as error:
            print(f"Could not load {file.name}: {error}")

print("\nLOADED SUPPORTING DATASETS:",
      len(supporting_data))

print("\nAVAILABLE DATASET NAMES:")
for name in supporting_data:
    print("-", name)

print("\nSUPPORTING DATASET DISCOVERY COMPLETED")

DISCOVERING SUPPORTING DATASETS


,File,Column Count,Columns
0,factory_master.csv,3,"[Factory_Name, Latitude, Longitude]"
1,high_derived_lead_time_risk.csv,12,"[Product Name, Order_Lines, Unique_Orders, Mea..."
2,kpi_average_derived_lead_time.csv,7,"[KPI, Value, Unit, Scope, Valid_Record_Count, ..."
3,kpi_dashboard_data.csv,6,"[Metric, Value, Unit, Scope, Status, Interpret..."
4,kpi_estimated_lead_time_reduction.csv,7,"[KPI, Value, Unit, Alternative_Options_Checked..."
5,kpi_export_manifest.csv,4,"[File, Rows, Columns, Status]"
6,kpi_gross_profit.csv,7,"[KPI, Value, Unit, Scope, Record_Count, Status..."
7,kpi_lead_time_variability.csv,5,"[KPI, Value, Unit, Method, Interpretation]"
8,kpi_median_derived_lead_time.csv,7,"[KPI, Value, Unit, Scope, Valid_Record_Count, ..."
9,kpi_profit_margin.csv,6,"[KPI, Value, Unit, Scope, Status, Interpretation]"



LOADED SUPPORTING DATASETS: 46

AVAILABLE DATASET NAMES:
- factory_master
- high_derived_lead_time_risk
- kpi_average_derived_lead_time
- kpi_dashboard_data
- kpi_estimated_lead_time_reduction
- kpi_export_manifest
- kpi_gross_profit
- kpi_lead_time_variability
- kpi_median_derived_lead_time
- kpi_profit_margin
- kpi_total_sales
- low_margin_risk
- low_sample_size_risk
- model_uncertainty_report
- phase11_integrity_report
- profit_stability_proxy
- recommendation_coverage
- region_performance_risk
- region_sample_size_risk
- risk_register
- scenario_support_score
- processed_dataset
- product_factory_mapping
- application_runtime_report
- application_validation_report
- complete_application_validation
- metrics_export_manifest
- ML_Cross_Validation
- ML_Dashboard_Metrics
- ML_Final_Model_Comparison
- ML_LeadTime_Ranges
- ML_Predictions
- phase_12_validation_report
- Phase_10_Factory_Option_Screening
- Phase_10_Product_Review_Priorities
- ranked_product_recommendations
- recommendation

In [78]:
# Inspect available model files and metadata
# This cell checks which trained models and configuration files actually exist. It does not assume a particular model filename or load an unverified model

print("INSPECTING MODEL FILES AND METADATA")

print("\nMODEL FILES")

if model_files:
    for file in model_files:
        print(
            file.relative_to(APP_ROOT),
            "| Size:",
            round(file.stat().st_size / 1024, 2),
            "KB"
        )
else:
    print("No joblib or pickle model files found.")

print("\nJSON CONFIGURATION FILES")

if json_files:
    for file in json_files:
        print("\nFile:", file.relative_to(APP_ROOT))

        try:
            with open(file, "r", encoding="utf-8") as handle:
                metadata_content = json.load(handle)

            if isinstance(metadata_content, dict):
                print("Available keys:")
                for key in metadata_content.keys():
                    print("-", key)

            elif isinstance(metadata_content, list):
                print("List entries:", len(metadata_content))

            else:
                print("Content type:",
                      type(metadata_content).__name__)

        except Exception as error:
            print("Could not read JSON:", error)
else:
    print("No JSON configuration files found.")

print("\nMODEL AND METADATA INSPECTION COMPLETED")

INSPECTING MODEL FILES AND METADATA

MODEL FILES
models\Nassau_Lead_Time_Model.joblib | Size: 19266.72 KB

JSON CONFIGURATION FILES

File: models\feature_schema.json
Available keys:
- target_variable
- predictor_features
- feature_count
- selected_model

File: models\model_metadata.json
Available keys:
- project
- target_variable
- selected_model
- predictor_features
- training_records
- testing_records
- test_MAE
- test_RMSE
- test_R2
- model_file

File: reports\model_metrics\metrics_review_note.json
Available keys:
- review_status
- note
- metadata_metrics
- comparison_file
- comparison_values_for_tuned_random_forest
- cross_validation_note

File: reports\model_metrics\model_metadata.json
Available keys:
- project
- target_variable
- selected_model
- predictor_features
- training_records
- testing_records
- test_MAE
- test_RMSE
- test_R2
- model_file

MODEL AND METADATA INSPECTION COMPLETED


In [80]:
# Verify original dataset integrity

import hashlib

print("VERIFYING ORIGINAL DATA INTEGRITY")

expected_sha256 = (
    "6ae9208e77ce410e90073b19813d29f89853ce9f3cce43d41f9643781696e94f"
)

raw_candidates = []

for file in PROJECT_ROOT.rglob("*.csv"):
    if "app_bundle" in file.parts:
        continue

    try:
        sample = pd.read_csv(file, nrows=3)

        if {"Order Date", "Ship Date", "Sales"}.issubset(sample.columns):
            raw_candidates.append(file)

    except Exception:
        pass

if raw_candidates:
    for file in raw_candidates:
        digest = hashlib.sha256(file.read_bytes()).hexdigest()

        print("\nFile:", file)
        print("SHA256:", digest)
        print("Matches expected hash:", digest == expected_sha256)
else:
    print("No matching original CSV identified.")

print("\nINTEGRITY CHECK COMPLETED")

VERIFYING ORIGINAL DATA INTEGRITY

File: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\Nassau_Candy_Factory_Mapped_Reference.csv
SHA256: 0b8b79e98bc9665bf2ef1c5fe4d289346e4d1a3591d67e711021dac61a822b8c
Matches expected hash: False

File: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\Nassau_Candy_Feature_Engineered.csv
SHA256: 7b4f332a3ea789c48c46752246a83812b2d83b932f41e2ba96569be067ce790f
Matches expected hash: False

File: C:\Users\hp\Nassau_Candy_Final_Submission\data\raw\Nassau Candy Distributor.csv
SHA256: 6ae9208e77ce410e90073b19813d29f89853ce9f3cce43d41f9643781696e94f
Matches expected hash: True

INTEGRITY CHECK COMPLETED


In [82]:
# Check required Python packages

import importlib.util

required_packages = [
    "pandas",
    "numpy",
    "streamlit",
    "plotly",
    "openpyxl",
    "joblib",
    "sklearn"
]

package_status = []

for package in required_packages:
    installed = importlib.util.find_spec(package) is not None

    package_status.append({
        "Package": package,
        "Installed": installed
    })

package_status_df = pd.DataFrame(package_status)

display(package_status_df)

missing_packages = package_status_df.loc[
    ~package_status_df["Installed"],
    "Package"
].tolist()

if missing_packages:
    print("\nMissing packages:", missing_packages)
    print("Install using:")
    print("%pip install " + " ".join(missing_packages))
else:
    print("\nALL REQUIRED PACKAGES ARE AVAILABLE")

,Package,Installed
0,pandas,True
1,numpy,True
2,streamlit,True
3,plotly,True
4,openpyxl,True
5,joblib,True
6,sklearn,True



ALL REQUIRED PACKAGES ARE AVAILABLE


In [84]:
# Import application dependencies 

import os
import sys
import json
import warnings
import textwrap
import joblib

import pandas as pd
import numpy as np

import streamlit as st
import plotly.express as px
import plotly.graph_objects as go

from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore", category=FutureWarning)

print("APPLICATION DEPENDENCIES IMPORTED")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Streamlit:", st.__version__)
print("Plotly:", px.__version__ if hasattr(px, "__version__") else "Available")

APPLICATION DEPENDENCIES IMPORTED
Pandas: 2.2.2
NumPy: 1.26.4
Streamlit: 1.32.0
Plotly: Available


In [86]:
# Identify dashboard data sources

print("IDENTIFYING DASHBOARD DATA SOURCES")

dashboard_sources = {
    "transactions": PROCESSED_DATA_PATH
}

for name, dataframe in supporting_data.items():
    print("\nDATASET:", name)
    print("Shape:", dataframe.shape)
    print("Columns:", dataframe.columns.tolist())

print("\nTRANSACTION DATA:")
print(dashboard_sources["transactions"])

print("\nSOURCE IDENTIFICATION COMPLETED")

IDENTIFYING DASHBOARD DATA SOURCES

DATASET: factory_master
Shape: (5, 3)
Columns: ['Factory_Name', 'Latitude', 'Longitude']

DATASET: high_derived_lead_time_risk
Shape: (15, 12)
Columns: ['Product Name', 'Order_Lines', 'Unique_Orders', 'Mean_Lead_Time_Days', 'Median_Lead_Time_Days', 'Lead_Time_Std_Days', 'Lead_Time_IQR_Days', 'Threshold_Days', 'Sample_Eligible', 'High_Lead_Time_Risk_Flag', 'Risk_Basis', 'Limitation']

DATASET: kpi_average_derived_lead_time
Shape: (1, 7)
Columns: ['KPI', 'Value', 'Unit', 'Scope', 'Valid_Record_Count', 'Status', 'Interpretation']

DATASET: kpi_dashboard_data
Shape: (9, 6)
Columns: ['Metric', 'Value', 'Unit', 'Scope', 'Status', 'Interpretation']

DATASET: kpi_estimated_lead_time_reduction
Shape: (1, 7)
Columns: ['KPI', 'Value', 'Unit', 'Alternative_Options_Checked', 'Options_With_Estimable_Factory_Lead_Time', 'Status', 'Interpretation']

DATASET: kpi_export_manifest
Shape: (19, 4)
Columns: ['File', 'Rows', 'Columns', 'Status']

DATASET: kpi_gross_profit


In [88]:
# Prepare application-ready data 

app_transactions = dashboard_data.copy()

app_transactions = app_transactions.dropna(
    subset=["Product Name", "Region", "Ship Mode"]
)

for column in ["Sales", "Units", "Gross Profit", "Cost"]:
    app_transactions[column] = pd.to_numeric(
        app_transactions[column],
        errors="coerce"
    )

app_transactions["Sales"] = app_transactions["Sales"].fillna(0)
app_transactions["Units"] = app_transactions["Units"].fillna(0)
app_transactions["Gross Profit"] = (
    app_transactions["Gross Profit"].fillna(0)
)
app_transactions["Cost"] = app_transactions["Cost"].fillna(0)

app_transactions["Profit Margin"] = np.where(
    app_transactions["Sales"] != 0,
    app_transactions["Gross Profit"] /
    app_transactions["Sales"] * 100,
    np.nan
)

print("APPLICATION DATA PREPARED")
print("Rows:", len(app_transactions))
print("Products:", app_transactions["Product Name"].nunique())
print("Regions:", app_transactions["Region"].nunique())
print("Ship modes:", app_transactions["Ship Mode"].nunique())

display(app_transactions.head())

APPLICATION DATA PREPARED
Rows: 10194
Products: 15
Regions: 4
Ship modes: 4


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,...,Sales_Per_Unit,Cost_Per_Unit,Profit_Per_Unit,Lead_Time_Category,Profit_Category,Sales_Category,_Order_Date,_Ship_Date,_Lead_Time_Days,Profit Margin
0,1,US-2021-103800-CHO-MIL-31000,03-01-2024,30-06-2026,Standard Class,103800,United States,Houston,Texas,77095,...,3.25,1.14,2.11,Low,Low,Low,2024-01-03,2026-06-30,909,64.923077
1,2,US-2021-112326-CHO-TRI-54000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,...,3.75,1.30,2.45,Low,Low,Low,2024-01-04,2026-07-01,909,65.333333
2,3,US-2021-112326-CHO-NUT-13000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,...,3.49,1.00,2.49,Low,Medium,Medium,2024-01-04,2026-07-01,909,71.346705
3,4,US-2021-112326-CHO-SCR-58000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,...,3.60,1.10,2.50,Low,Medium,Medium,2024-01-04,2026-07-01,909,69.444444
4,5,US-2021-141817-CHO-TRI-54000,05-01-2024,05-07-2026,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,...,3.75,1.30,2.45,Low,Medium,Medium,2024-01-05,2026-07-05,912,65.333333


In [90]:
# Create reusable KPI calculations

def calculate_dashboard_kpis(data):
    if data.empty:
        return {
            "Total Sales": 0,
            "Total Profit": 0,
            "Total Units": 0,
            "Average Lead Time": np.nan,
            "Profit Margin": np.nan,
            "Orders": 0
        }

    total_sales = data["Sales"].sum()
    total_profit = data["Gross Profit"].sum()

    return {
        "Total Sales": total_sales,
        "Total Profit": total_profit,
        "Total Units": data["Units"].sum(),
        "Average Lead Time": data["_Lead_Time_Days"].mean(),
        "Profit Margin": (
            total_profit / total_sales * 100
            if total_sales != 0 else np.nan
        ),
        "Orders": data["Order ID"].nunique()
    }


kpi_summary = calculate_dashboard_kpis(app_transactions)

display(pd.DataFrame(
    kpi_summary.items(),
    columns=["KPI", "Value"]
))

print("KPI CALCULATIONS COMPLETED")

,KPI,Value
0,Total Sales,141783.630000
1,Total Profit,93442.800000
2,Total Units,38654.000000
3,Average Lead Time,1320.841868
4,Profit Margin,65.905211
5,Orders,8549.000000


KPI CALCULATIONS COMPLETED


In [92]:
# Identify available recommendation datasets 

print("IDENTIFYING RECOMMENDATION DATA")

recommendation_candidates = {}

for name, dataframe in supporting_data.items():
    combined = (
        name + " " + " ".join(dataframe.columns.astype(str))
    ).lower()

    if any(word in combined for word in [
        "recommend", "priority", "action", "opportunity"
    ]):
        recommendation_candidates[name] = dataframe

for name, dataframe in recommendation_candidates.items():
    print("\nRecommendation source:", name)
    print("Shape:", dataframe.shape)
    print("Columns:", dataframe.columns.tolist())
    display(dataframe.head(3))

print("\nRECOMMENDATION DISCOVERY COMPLETED")

IDENTIFYING RECOMMENDATION DATA

Recommendation source: recommendation_coverage
Shape: (7, 6)
Columns: ['Coverage_Metric', 'Count', 'Interpretation', 'Coverage_Percent', 'Coverage_Source', 'Interpretation_Limit']


,Coverage_Metric,Count,Interpretation,Coverage_Percent,Coverage_Source,Interpretation_Limit
0,Source products,15,Products present in the original CSV,100.0,Phase 10 task_10_17_coverage_kpis.csv,Product reference mapping is not proof of hist...
1,Products with reference mapping,15,Products with a Phase 07 reference mapping,100.0,Phase 10 task_10_17_coverage_kpis.csv,Product reference mapping is not proof of hist...
2,Unmapped products,0,Products requiring mapping review,0.0,Phase 10 task_10_17_coverage_kpis.csv,Product reference mapping is not proof of hist...



Recommendation source: risk_register
Shape: (16, 11)
Columns: ['Risk_ID', 'Category', 'Scope', 'Indicator', 'Threshold', 'Observed_Value', 'Severity', 'Evidence_Type', 'Evidence', 'Limitation', 'Suggested_Action_or_Data_Needed']


,Risk_ID,Category,Scope,Indicator,Threshold,Observed_Value,Severity,Evidence_Type,Evidence,Limitation,Suggested_Action_or_Data_Needed
0,LT-001,High derived lead time,Product: Laffy Taffy,Mean derived lead-time days,1361.1191489361702,1383.1,Review,Historical descriptive signal,Product mean derived lead-time at or above the...,Descriptive historical flag only. Derived date...,Validate source date definitions and obtain ac...
1,LT-002,High derived lead time,Product: SweeTARTS,Mean derived lead-time days,1361.1191489361702,1383.5,Review,Historical descriptive signal,Product mean derived lead-time at or above the...,Descriptive historical flag only. Derived date...,Validate source date definitions and obtain ac...
2,LM-003,Low historical gross margin,Product: Kazookles,Historical gross margin,48.33333333333333,7.6923076923076925,Review,Historical descriptive signal,Product gross profit divided by product sales;...,"Does not include unrecorded overhead, future p...","Review product pricing, recorded cost componen..."



Recommendation source: Phase_10_Factory_Option_Screening
Shape: (75, 15)
Columns: ['Product Name', 'Reference Factory', 'Factory Option', 'Is_Reference_Mapping', 'Option_Type', 'Compatibility_Status', 'Factory_Specific_Lead_Time_Estimable', 'Factory_Specific_Cost_Available', 'Capacity_Data_Available', 'Speed_Priority_Score', 'Profit_Priority_Score', 'Balanced_Score', 'Screening_Rank', 'Factory_Option_Rank_Status', 'Recommendation_Status']


,Product Name,Reference Factory,Factory Option,Is_Reference_Mapping,Option_Type,Compatibility_Status,Factory_Specific_Lead_Time_Estimable,Factory_Specific_Cost_Available,Capacity_Data_Available,Speed_Priority_Score,Profit_Priority_Score,Balanced_Score,Screening_Rank,Factory_Option_Rank_Status,Recommendation_Status
0,SweeTARTS,Sugar Shack,Lot's O' Nuts,False,Alternative factory - unverified,UNVERIFIED,False,False,False,0.607029,0.460993,0.534011,1,Not a feasibility or factory-benefit rank,UNVERIFIED - do not treat as an approved alter...
1,SweeTARTS,Sugar Shack,Secret Factory,False,Alternative factory - unverified,UNVERIFIED,False,False,False,0.607029,0.460993,0.534011,1,Not a feasibility or factory-benefit rank,UNVERIFIED - do not treat as an approved alter...
2,SweeTARTS,Sugar Shack,Sugar Shack,True,Reference mapping only - not verified historic...,Reference mapping; actual dispatch not verified,False,False,False,0.607029,0.460993,0.534011,1,Product-level screening rank only,Reference mapping for review


... (13 outputs left out)

,Product Name,Speed_Priority_Score,Profit_Priority_Score,Balanced_Score,Order_Line_Count,Historical_Sales,Historical_Gross_Profit,Historical_Gross_Margin,Derived_Lead_Time_Mean_Days,Lead_Time_Std_Days,Lead_Time_IQR_Days,Review_Priority_Reason,Important_Qualification,Explanation_Data_Status
0,SweeTARTS,0.607029,0.460993,0.534011,10,61.50,28.70,0.466667,1383.500000,246.688850,366.50,Higher-than-median derived lead time; Lower-th...,Product-level review priority only. Derived da...,Historical indicators available
1,Hair Toffee,1.000000,0.030733,0.515366,4,76.50,59.50,0.777778,1455.250000,209.876750,361.75,Higher-than-median derived lead time; At-or-ab...,Product-level review priority only. Derived da...,Historical indicators available
2,Kazookles,0.001769,1.000000,0.500884,96,1205.75,92.75,0.076923,1272.989583,259.943212,186.50,At-or-below-median derived lead time; Lower-th...,Product-level review priority only. Derived da...,Historical indicators available



Recommendation source: task_10_22_business_action_plan
Shape: (7, 6)
Columns: ['Priority', 'Business_Action', 'Evidence_From_Analysis', 'Suggested_Owner', 'Next_Data_Required', 'Expected_Decision']


,Priority,Business_Action,Evidence_From_Analysis,Suggested_Owner,Next_Data_Required,Expected_Decision
0,1,Investigate the unusually long derived Order D...,Historical date differences are derived direct...,Order Management / Data Governance,"Confirm date-field definitions, order lifecycl...",Determine whether derived lead-time days repre...
1,2,Review products with the highest speed-priorit...,Product-level historical derived lead-time ran...,Supply Chain Planning,"Validated production start/end dates, dispatch...",Identify products that merit operational inves...
2,3,Review products with lower historical gross ma...,Historical product-level gross margin and prof...,Finance / Commercial Analytics,"Verified manufacturing cost, freight cost, dis...",Validate margin drivers before changing prices...



RECOMMENDATION DISCOVERY COMPLETED


In [94]:
# Identify available risk datasets

print("IDENTIFYING RISK DATA")

risk_candidates = {}

for name, dataframe in supporting_data.items():
    combined = (
        name + " " + " ".join(dataframe.columns.astype(str))
    ).lower()

    if any(word in combined for word in [
        "risk", "exposure", "sensitivity", "impact"
    ]):
        risk_candidates[name] = dataframe

for name, dataframe in risk_candidates.items():
    print("\nRisk source:", name)
    print("Shape:", dataframe.shape)
    print("Columns:", dataframe.columns.tolist())
    display(dataframe.head(3))

print("\nRISK DATA DISCOVERY COMPLETED")

IDENTIFYING RISK DATA

Risk source: high_derived_lead_time_risk
Shape: (15, 12)
Columns: ['Product Name', 'Order_Lines', 'Unique_Orders', 'Mean_Lead_Time_Days', 'Median_Lead_Time_Days', 'Lead_Time_Std_Days', 'Lead_Time_IQR_Days', 'Threshold_Days', 'Sample_Eligible', 'High_Lead_Time_Risk_Flag', 'Risk_Basis', 'Limitation']


,Product Name,Order_Lines,Unique_Orders,Mean_Lead_Time_Days,Median_Lead_Time_Days,Lead_Time_Std_Days,Lead_Time_IQR_Days,Threshold_Days,Sample_Eligible,High_Lead_Time_Risk_Flag,Risk_Basis,Limitation
0,Everlasting Gobstopper,3,3,1394.666667,1273.0,213.336198,185.5,1361.119149,False,False,Product mean derived lead-time at or above the...,Descriptive historical flag only. Derived date...
1,Fizzy Lifting Drinks,6,6,1273.500000,1274.0,326.691751,546.0,1361.119149,True,False,Product mean derived lead-time at or above the...,Descriptive historical flag only. Derived date...
2,Fun Dip,3,3,1272.666667,1272.0,2.081666,2.0,1361.119149,False,False,Product mean derived lead-time at or above the...,Descriptive historical flag only. Derived date...



Risk source: low_margin_risk
Shape: (15, 12)
Columns: ['Product Name', 'Order_Lines', 'Unique_Orders', 'Total_Sales', 'Total_Gross_Profit', 'Gross_Margin_Pct', 'Break_Even_Risk_Flag', 'Lower_Quartile_Margin_Flag', 'Sample_Eligible', 'Low_Margin_Risk_Flag', 'Lower_Quartile_Threshold_Pct', 'Interpretation']


,Product Name,Order_Lines,Unique_Orders,Total_Sales,Total_Gross_Profit,Gross_Margin_Pct,Break_Even_Risk_Flag,Lower_Quartile_Margin_Flag,Sample_Eligible,Low_Margin_Risk_Flag,Lower_Quartile_Threshold_Pct,Interpretation
0,Everlasting Gobstopper,3,3,130.00,104.00,80.0,False,False,False,False,48.333333,Break-even and dataset-relative margin flags a...
1,Fizzy Lifting Drinks,6,6,78.75,47.25,60.0,False,False,True,False,48.333333,Break-even and dataset-relative margin flags a...
2,Fun Dip,3,3,12.00,4.80,40.0,False,True,False,False,48.333333,Break-even and dataset-relative margin flags a...



Risk source: low_sample_size_risk
Shape: (15, 6)
Columns: ['Product Name', 'Order_Lines', 'Unique_Orders', 'Sample_Support_Category', 'Low_Sample_Size_Risk_Flag', 'Threshold_Explanation']


,Product Name,Order_Lines,Unique_Orders,Sample_Support_Category,Low_Sample_Size_Risk_Flag,Threshold_Explanation
0,Everlasting Gobstopper,3,3,Low sample,True,Screening rule: 10 or fewer distinct orders = ...
1,Fizzy Lifting Drinks,6,6,Low sample,True,Screening rule: 10 or fewer distinct orders = ...
2,Fun Dip,3,3,Low sample,True,Screening rule: 10 or fewer distinct orders = ...


... (7 outputs left out)

,Risk_ID,Category,Scope,Indicator,Threshold,Observed_Value,Severity,Evidence_Type,Evidence,Limitation,Suggested_Action_or_Data_Needed
0,LT-001,High derived lead time,Product: Laffy Taffy,Mean derived lead-time days,1361.1191489361702,1383.1,Review,Historical descriptive signal,Product mean derived lead-time at or above the...,Descriptive historical flag only. Derived date...,Validate source date definitions and obtain ac...
1,LT-002,High derived lead time,Product: SweeTARTS,Mean derived lead-time days,1361.1191489361702,1383.5,Review,Historical descriptive signal,Product mean derived lead-time at or above the...,Descriptive historical flag only. Derived date...,Validate source date definitions and obtain ac...
2,LM-003,Low historical gross margin,Product: Kazookles,Historical gross margin,48.33333333333333,7.6923076923076925,Review,Historical descriptive signal,Product gross profit divided by product sales;...,"Does not include unrecorded overhead, future p...","Review product pricing, recorded cost componen..."



Risk source: ranked_product_recommendations
Shape: (45, 35)
Columns: ['Product Name', 'Historical_Rows', 'Historical_Units', 'Historical_Sales', 'Historical_Cost', 'Historical_Gross_Profit', 'Median_Lead_Time_Days', 'Mean_Lead_Time_Days', 'Lead_Time_Std_Days', 'Lead_Time_IQR_Days', 'Historical_Margin_Pct', 'Mapped Factory', 'Mapping_Status', 'Sample_Support', 'Eligible_For_Descriptive_Scoring', 'Additional_Cost_Per_Unit', 'Hypothetical_Additional_Cost_Per_Unit', 'Hypothetical_Additional_Cost_Total', 'Cost_Assumption_Status', 'Lead_Time_Priority_Score', 'Variability_Priority_Score', 'Margin_Risk_Score', 'Added_Cost_Risk_Score', 'Profit_Protection_Score', 'Speed_First_Score', 'Balanced_Score', 'Scoring_Interpretation', 'Factory_Recommendation_Status', 'Rank', 'Priority_Profile', 'Review_Priority_Score', 'Recommendation_Rationale', 'Baseline_Factory_Reference', 'Alternative_Factory_Status', 'Recommendation_Type']


,Product Name,Historical_Rows,Historical_Units,Historical_Sales,Historical_Cost,Historical_Gross_Profit,Median_Lead_Time_Days,Mean_Lead_Time_Days,Lead_Time_Std_Days,Lead_Time_IQR_Days,...,Balanced_Score,Scoring_Interpretation,Factory_Recommendation_Status,Rank,Priority_Profile,Review_Priority_Score,Recommendation_Rationale,Baseline_Factory_Reference,Alternative_Factory_Status,Recommendation_Type
0,Nerds,4,10,15.00,8.00,7.00,1274.0,1274.0,422.043442,730.5,...,73.383333,Relative product review priority; not an optim...,No alternative-factory recommendation supporte...,1,Speed_First,72.966667,High historical lead-time variability; investi...,Sugar Shack,Not recommended: actual factory capability and...,"Product-level review priority, not factory rea..."
1,Fizzy Lifting Drinks,6,21,78.75,31.50,47.25,1274.0,1273.5,326.691751,546.0,...,66.000000,Relative product review priority; not an optim...,No alternative-factory recommendation supporte...,2,Speed_First,69.333333,High historical lead-time variability; investi...,Sugar Shack,Not recommended: actual factory capability and...,"Product-level review priority, not factory rea..."
2,Laffy Taffy,10,27,53.73,20.25,33.48,1275.0,1383.1,246.563695,365.0,...,61.033333,Relative product review priority; not an optim...,No alternative-factory recommendation supporte...,3,Speed_First,68.866667,High historical derived lead time; validate da...,Sugar Shack,Not recommended: actual factory capability and...,"Product-level review priority, not factory rea..."



RISK DATA DISCOVERY COMPLETED


In [96]:
# Create application configuration

APP_CONFIG = {
    "title": "Nassau Candy Analytics",
    "page_icon": "🍬",
    "layout": "wide",
    "data_path": str(PROCESSED_DATA_PATH),
    "model_available": bool(model_files),
    "products": sorted(
        app_transactions["Product Name"].dropna().unique().tolist()
    ),
    "regions": sorted(
        app_transactions["Region"].dropna().unique().tolist()
    ),
    "ship_modes": sorted(
        app_transactions["Ship Mode"].dropna().unique().tolist()
    )
}

APP_CONFIG_PATH = APP_ROOT / "app_config.json"

with open(APP_CONFIG_PATH, "w", encoding="utf-8") as file:
    json.dump(APP_CONFIG, file, indent=4)

print("APPLICATION CONFIGURATION SAVED")
print(APP_CONFIG_PATH)

APPLICATION CONFIGURATION SAVED
C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\app_config.json


In [102]:
# Verify application file and configuration
print("VERIFYING APPLICATION OUTPUTS")

for file in [
    APP_FILE,
    APP_CONFIG_PATH
]:
    print(
        file.name,
        "| Exists:", file.exists(),
        "| Size:", file.stat().st_size if file.exists() else 0
    )

if not APP_FILE.exists():
    raise FileNotFoundError("Application source was not created.")

if not APP_CONFIG_PATH.exists():
    raise FileNotFoundError("Application configuration was not created.")

print("\nAPPLICATION OUTPUT VERIFICATION COMPLETED")

VERIFYING APPLICATION OUTPUTS
app.py | Exists: True | Size: 14914
app_config.json | Exists: True | Size: 972

APPLICATION OUTPUT VERIFICATION COMPLETED


In [125]:
import py_compile

print("CHECKING APPLICATION SYNTAX")

py_compile.compile(
    str(APP_FILE),
    doraise=True
)

print("APPLICATION SYNTAX: PASS")

CHECKING APPLICATION SYNTAX
APPLICATION SYNTAX: PASS


In [98]:
# Create the complete Streamlit application

app_source = r'''
import json
from pathlib import Path

import numpy as np
import pandas as pd
import streamlit as st
import plotly.express as px
import plotly.graph_objects as go

st.set_page_config(
    page_title="Nassau Candy Analytics",
    page_icon="🍬",
    layout="wide"
)

BASE = Path(__file__).resolve().parent
DATA_DIR = BASE / "data"
REPORT_DIR = BASE / "reports"

st.markdown("""
<style>
.main {background-color: #f7f8fc;}
.block-container {padding-top: 1.5rem;}
[data-testid="stMetric"] {
    background: white;
    padding: 16px;
    border-radius: 12px;
    border: 1px solid #e7eaf0;
}
</style>
""", unsafe_allow_html=True)


@st.cache_data
def load_transactions():
    config_path = BASE / "app_config.json"

    with open(config_path, encoding="utf-8") as f:
        config = json.load(f)

    path = Path(config["data_path"])

    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")

    df = pd.read_csv(path)

    df["_Order_Date"] = pd.to_datetime(
        df["Order Date"], dayfirst=True, errors="coerce"
    )
    df["_Ship_Date"] = pd.to_datetime(
        df["Ship Date"], dayfirst=True, errors="coerce"
    )
    df["_Lead_Time_Days"] = (
        df["_Ship_Date"] - df["_Order_Date"]
    ).dt.days

    for col in ["Sales", "Units", "Gross Profit", "Cost"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


@st.cache_data
def load_supporting_files():
    result = {}

    for path in DATA_DIR.rglob("*.csv"):
        try:
            result[path.stem] = pd.read_csv(path)
        except Exception:
            continue

    return result


def money(value):
    return f"${value:,.2f}"


def safe_mean(series):
    return series.mean() if series.notna().any() else np.nan


def show_downloads(data):
    st.subheader("Download analysis")

    st.download_button(
        "Download filtered data (CSV)",
        data.to_csv(index=False).encode("utf-8"),
        "nassau_filtered_analysis.csv",
        "text/csv"
    )

    output = data.to_csv(index=False).encode("utf-8")

    st.download_button(
        "Download analysis report (CSV)",
        output,
        "nassau_analysis_report.csv",
        "text/csv"
    )


try:
    df = load_transactions()
except Exception as error:
    st.error(f"Unable to load application data: {error}")
    st.stop()


required = [
    "Order ID", "Product Name", "Region", "Ship Mode",
    "Sales", "Units", "Gross Profit", "Cost"
]

missing = [c for c in required if c not in df.columns]

if missing:
    st.error(f"Required dataset columns are missing: {missing}")
    st.stop()


supporting = load_supporting_files()

st.title("🍬 Nassau Candy Analytics")
st.caption("Sales, profitability, shipping and operational analysis")

pages = [
    "Home",
    "Product Performance",
    "Region Analysis",
    "Ship Mode Analysis",
    "Factory Explorer",
    "Scenario Simulator",
    "Recommendations",
    "Risk and Impact"
]

page = st.sidebar.radio("Navigation", pages)

st.sidebar.header("Dashboard Filters")

products = sorted(df["Product Name"].dropna().unique())
regions = sorted(df["Region"].dropna().unique())
ship_modes = sorted(df["Ship Mode"].dropna().unique())

selected_products = st.sidebar.multiselect(
    "Products", products, default=products
)
selected_regions = st.sidebar.multiselect(
    "Regions", regions, default=regions
)
selected_modes = st.sidebar.multiselect(
    "Ship modes", ship_modes, default=ship_modes
)

if not selected_products or not selected_regions or not selected_modes:
    st.warning("Your filters returned no selections. Select at least one value in each filter.")
    st.stop()

filtered = df[
    df["Product Name"].isin(selected_products)
    & df["Region"].isin(selected_regions)
    & df["Ship Mode"].isin(selected_modes)
].copy()

if filtered.empty:
    st.warning("No records match the selected filters. Adjust your filters.")
    st.stop()

sales = filtered["Sales"].sum()
profit = filtered["Gross Profit"].sum()
units = filtered["Units"].sum()
orders = filtered["Order ID"].nunique()
lead_time = safe_mean(filtered["_Lead_Time_Days"])
margin = profit / sales * 100 if sales else np.nan


if page == "Home":
    st.header("Business Performance Overview")

    c1, c2, c3, c4 = st.columns(4)

    c1.metric("Total Sales", money(sales))
    c2.metric("Gross Profit", money(profit))
    c3.metric("Units Sold", f"{units:,.0f}")
    c4.metric("Orders", f"{orders:,}")

    c5, c6 = st.columns(2)
    c5.metric(
        "Profit Margin",
        f"{margin:.2f}%" if np.isfinite(margin) else "Not available"
    )
    c6.metric(
        "Average Derived Lead Time",
        f"{lead_time:.2f} days" if np.isfinite(lead_time) else "Not available"
    )

    left, right = st.columns(2)

    with left:
        product_sales = (
            filtered.groupby("Product Name", as_index=False)["Sales"]
            .sum().sort_values("Sales", ascending=False)
        )
        st.plotly_chart(
            px.bar(product_sales, x="Product Name", y="Sales",
                   title="Sales by Product"),
            use_container_width=True
        )

    with right:
        region_sales = (
            filtered.groupby("Region", as_index=False)["Sales"]
            .sum().sort_values("Sales", ascending=False)
        )
        st.plotly_chart(
            px.bar(region_sales, x="Region", y="Sales",
                   title="Sales by Region"),
            use_container_width=True
        )


elif page == "Product Performance":
    st.header("Product Performance")

    product_summary = (
        filtered.groupby("Product Name", as_index=False)
        .agg(
            Sales=("Sales", "sum"),
            Gross_Profit=("Gross Profit", "sum"),
            Units=("Units", "sum"),
            Orders=("Order ID", "nunique")
        )
    )

    product_summary["Profit Margin (%)"] = np.where(
        product_summary["Sales"] != 0,
        product_summary["Gross_Profit"] /
        product_summary["Sales"] * 100,
        np.nan
    )

    st.dataframe(
        product_summary.sort_values("Sales", ascending=False),
        use_container_width=True
    )

    st.plotly_chart(
        px.bar(
            product_summary.sort_values("Sales", ascending=False),
            x="Product Name", y="Sales",
            title="Product Sales Comparison"
        ),
        use_container_width=True
    )

    st.plotly_chart(
        px.scatter(
            product_summary,
            x="Sales",
            y="Gross_Profit",
            size="Units",
            hover_name="Product Name",
            title="Product Sales and Profit Relationship"
        ),
        use_container_width=True
    )


elif page == "Region Analysis":
    st.header("Regional Analysis")

    region_summary = (
        filtered.groupby("Region", as_index=False)
        .agg(
            Sales=("Sales", "sum"),
            Gross_Profit=("Gross Profit", "sum"),
            Units=("Units", "sum"),
            Orders=("Order ID", "nunique")
        )
    )

    region_summary["Profit Margin (%)"] = np.where(
        region_summary["Sales"] != 0,
        region_summary["Gross_Profit"] /
        region_summary["Sales"] * 100,
        np.nan
    )

    st.dataframe(region_summary, use_container_width=True)

    st.plotly_chart(
        px.bar(region_summary, x="Region", y="Sales",
               title="Sales by Region"),
        use_container_width=True
    )


elif page == "Ship Mode Analysis":
    st.header("Shipping Analysis")

    ship_summary = (
        filtered.groupby("Ship Mode", as_index=False)
        .agg(
            Sales=("Sales", "sum"),
            Gross_Profit=("Gross Profit", "sum"),
            Orders=("Order ID", "nunique"),
            Average_Lead_Time=("_Lead_Time_Days", "mean")
        )
    )

    st.dataframe(ship_summary, use_container_width=True)

    st.plotly_chart(
        px.bar(ship_summary, x="Ship Mode", y="Orders",
               title="Orders by Shipping Mode"),
        use_container_width=True
    )

    st.plotly_chart(
        px.bar(ship_summary, x="Ship Mode",
               y="Average_Lead_Time",
               title="Average Derived Lead Time by Ship Mode"),
        use_container_width=True
    )

    st.caption(
        "Lead time is calculated from Ship Date minus Order Date. "
        "It is not independently verified carrier transit time."
    )


elif page == "Factory Explorer":
    st.header("Factory Reference Explorer")

    factory_files = {
        name: data for name, data in supporting.items()
        if "factory" in name.lower()
    }

    if not factory_files:
        st.info("No factory reference CSV was found in the application data folder.")
    else:
        for name, data in factory_files.items():
            st.subheader(name.replace("_", " ").title())
            st.dataframe(data, use_container_width=True)

    st.info(
        "Factory selection is for exploring reference information. "
        "The prediction model does not use factory as an input feature. "
        "Alternative factory savings are not established by this dashboard."
    )


elif page == "Scenario Simulator":
    st.header("Scenario Simulator")

    st.write(
        "Explore illustrative changes to sales, cost and profit. "
        "These controls do not represent a probability of success."
    )

    sales_change = st.slider(
        "Illustrative sales change (%)", -30, 50, 0
    )
    cost_change = st.slider(
        "Illustrative cost change (%)", -30, 50, 0
    )
    speed_profit_priority = st.slider(
        "Speed versus profit priority",
        0, 100, 50
    )

    scenario_sales = sales * (1 + sales_change / 100)
    scenario_cost = filtered["Cost"].sum() * (
        1 + cost_change / 100
    )
    scenario_profit = scenario_sales - scenario_cost

    a, b, c = st.columns(3)
    a.metric("Scenario Sales", money(scenario_sales))
    b.metric("Scenario Cost", money(scenario_cost))
    c.metric("Scenario Profit", money(scenario_profit))

    comparison = pd.DataFrame({
        "Metric": ["Sales", "Cost", "Profit"],
        "Baseline": [
            sales,
            filtered["Cost"].sum(),
            profit
        ],
        "Scenario": [
            scenario_sales,
            scenario_cost,
            scenario_profit
        ]
    })

    st.plotly_chart(
        px.bar(
            comparison.melt(
                id_vars="Metric",
                var_name="Case",
                value_name="Amount"
            ),
            x="Metric",
            y="Amount",
            color="Case",
            barmode="group",
            title="Baseline versus Scenario"
        ),
        use_container_width=True
    )

    st.write(
        f"Selected speed-profit priority: {speed_profit_priority}/100"
    )
    st.caption(
        "The priority slider records a preference setting; it does not "
        "calculate an optimized or validated recommendation."
    )


elif page == "Recommendations":
    st.header("Recommendation Dashboard")

    recommendation_files = {
        name: data for name, data in supporting.items()
        if any(term in name.lower() for term in [
            "recommend", "priority", "opportunity"
        ])
    }

    if recommendation_files:
        for name, data in recommendation_files.items():
            st.subheader(name.replace("_", " ").title())
            st.dataframe(data, use_container_width=True)

            st.download_button(
                f"Download {name}",
                data.to_csv(index=False).encode("utf-8"),
                f"{name}.csv",
                "text/csv",
                key=f"download_{name}"
            )
    else:
        st.info(
            "No recommendation export was detected. "
            "The dashboard will show descriptive product and region rankings."
        )

        product_ranking = (
            filtered.groupby("Product Name", as_index=False)
            .agg(
                Sales=("Sales", "sum"),
                Gross_Profit=("Gross Profit", "sum")
            )
            .sort_values("Gross_Profit", ascending=False)
        )

        st.subheader("Products ranked by observed gross profit")
        st.dataframe(product_ranking, use_container_width=True)

        st.caption(
            "Rankings are descriptive, not proof of optimal decisions."
        )

    st.download_button(
        "Download current product ranking",
        data=product_ranking.to_csv(index=False).encode("utf-8")
        if "product_ranking" in locals()
        else filtered.to_csv(index=False).encode("utf-8"),
        "nassau_recommendations.csv",
        "text/csv"
    )


elif page == "Risk and Impact":
    st.header("Risk and Impact Analysis")

    risk_files = {
        name: data for name, data in supporting.items()
        if any(term in name.lower() for term in [
            "risk", "sensitivity", "impact"
        ])
    }

    if risk_files:
        for name, data in risk_files.items():
            st.subheader(name.replace("_", " ").title())
            st.dataframe(data, use_container_width=True)
    else:
        st.info("No exported risk analysis file was detected.")

    st.subheader("Observed business exposure")

    region_exposure = (
        filtered.groupby("Region", as_index=False)
        .agg(
            Sales=("Sales", "sum"),
            Gross_Profit=("Gross Profit", "sum"),
            Orders=("Order ID", "nunique")
        )
    )

    st.dataframe(region_exposure, use_container_width=True)

    st.caption(
        "This panel presents observed sales and profit exposure. "
        "It does not claim a quantified probability of operational failure."
    )


st.divider()

st.subheader("Export filtered analysis")

st.download_button(
    "Download filtered analysis CSV",
    filtered.to_csv(index=False).encode("utf-8"),
    "nassau_filtered_analysis.csv",
    "text/csv"
)

st.download_button(
    "Download product analysis Excel",
    data=(
        filtered.groupby("Product Name", as_index=False)
        .agg(
            Sales=("Sales", "sum"),
            Gross_Profit=("Gross Profit", "sum"),
            Units=("Units", "sum")
        )
        .to_excel if False else b""
    ),
    file_name="nassau_product_analysis.xlsx",
    disabled=True
)

st.caption(
    "Analysis is based on the selected records. "
    "Recommendations are indicative and not proven optimal."
)
'''

APP_FILE.write_text(
    app_source,
    encoding="utf-8"
)

print("STREAMLIT APPLICATION SOURCE CREATED")
print("File:", APP_FILE)
print("Lines:", len(app_source.splitlines()))

STREAMLIT APPLICATION SOURCE CREATED
File: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\app.py
Lines: 546


In [123]:
app_text = APP_FILE.read_text(encoding="utf-8")

old_code = '''    st.download_button(
        "Download current product ranking",
        data=product_ranking.to_csv(index=False).encode("utf-8")
        if "product_ranking" in locals()
        else filtered.to_csv(index=False).encode("utf-8"),
        "nassau_recommendations.csv",
        "text/csv"
    )'''

new_code = '''    recommendation_download = (
        product_ranking.to_csv(index=False).encode("utf-8")
        if "product_ranking" in locals()
        else filtered.to_csv(index=False).encode("utf-8")
    )

    st.download_button(
        "Download current product ranking",
        data=recommendation_download,
        file_name="nassau_recommendations.csv",
        mime="text/csv"
    )'''

if old_code not in app_text:
    raise ValueError(
        "Expected faulty code was not found. "
        "No changes were made."
    )

app_text = app_text.replace(old_code, new_code, 1)

APP_FILE.write_text(app_text, encoding="utf-8")

print("RECOMMENDATION DOWNLOAD CODE CORRECTED")

RECOMMENDATION DOWNLOAD CODE CORRECTED


In [104]:
# Test application data loading
print("TESTING APPLICATION DATA")

test_data = pd.read_csv(PROCESSED_DATA_PATH)

assert not test_data.empty
assert "Product Name" in test_data.columns
assert "Region" in test_data.columns
assert "Ship Mode" in test_data.columns
assert "Sales" in test_data.columns
assert "Gross Profit" in test_data.columns

print("Rows:", len(test_data))
print("Products:", test_data["Product Name"].nunique())
print("Regions:", test_data["Region"].nunique())
print("Ship modes:", test_data["Ship Mode"].nunique())

print("\nAPPLICATION DATA TEST: PASS")

TESTING APPLICATION DATA
Rows: 10194
Products: 15
Regions: 4
Ship modes: 4

APPLICATION DATA TEST: PASS


In [106]:
# Verify dashboard filters 

print("TESTING DASHBOARD FILTER LOGIC")

test_product = test_data["Product Name"].dropna().iloc[0]
test_region = test_data["Region"].dropna().iloc[0]
test_ship_mode = test_data["Ship Mode"].dropna().iloc[0]

test_filtered = test_data[
    (test_data["Product Name"] == test_product)
    & (test_data["Region"] == test_region)
    & (test_data["Ship Mode"] == test_ship_mode)
]

print("Test product:", test_product)
print("Test region:", test_region)
print("Test ship mode:", test_ship_mode)
print("Matching rows:", len(test_filtered))

assert len(test_filtered) > 0

empty_test = test_data[
    test_data["Product Name"] == "__NO_SUCH_PRODUCT__"
]

assert empty_test.empty

print("Valid filter test: PASS")
print("Empty filter test: PASS")

TESTING DASHBOARD FILTER LOGIC
Test product: Wonka Bar - Milk Chocolate
Test region: Interior
Test ship mode: Standard Class
Matching rows: 306
Valid filter test: PASS
Empty filter test: PASS


In [108]:
# Test recommendation export 

print("TESTING RECOMMENDATION EXPORT")

recommendation_test = (
    app_transactions.groupby("Product Name", as_index=False)
    .agg(
        Sales=("Sales", "sum"),
        Gross_Profit=("Gross Profit", "sum"),
        Units=("Units", "sum")
    )
    .sort_values("Gross_Profit", ascending=False)
)

recommendation_test_path = APP_REPORTS / "product_recommendation_test.csv"

recommendation_test.to_csv(
    recommendation_test_path,
    index=False
)

check_recommendation = pd.read_csv(
    recommendation_test_path
)

assert not check_recommendation.empty

display(check_recommendation.head())

print("Recommendation export:", recommendation_test_path)
print("RECOMMENDATION EXPORT TEST: PASS")

TESTING RECOMMENDATION EXPORT


,Product Name,Sales,Gross_Profit,Units
0,Wonka Bar -Scrumdiddlyumptious,27874.80,19357.50,7743
1,Wonka Bar - Triple Dazzle Caramel,28485.00,18610.20,7596
2,Wonka Bar - Milk Chocolate,26867.75,17443.37,8267
3,Wonka Bar - Nutty Crunch Surprise,23574.95,16819.95,6755
4,Wonka Bar - Fudge Mallows,24890.40,16593.60,6914


Recommendation export: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\reports\product_recommendation_test.csv
RECOMMENDATION EXPORT TEST: PASS


In [110]:
# Test Excel report generation

print("TESTING EXCEL REPORT GENERATION")

excel_report_path = APP_REPORTS / "nassau_dashboard_analysis.xlsx"

with pd.ExcelWriter(
    excel_report_path,
    engine="openpyxl"
) as writer:

    app_transactions.to_excel(
        writer,
        sheet_name="Transactions",
        index=False
    )

    recommendation_test.to_excel(
        writer,
        sheet_name="Product Analysis",
        index=False
    )

    pd.DataFrame(
        kpi_summary.items(),
        columns=["KPI", "Value"]
    ).to_excel(
        writer,
        sheet_name="KPI Summary",
        index=False
    )

print("Excel report saved:", excel_report_path)
print("File size:", round(excel_report_path.stat().st_size / 1024, 2), "KB")
print("EXCEL REPORT TEST: PASS")

TESTING EXCEL REPORT GENERATION
Excel report saved: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\reports\nassau_dashboard_analysis.xlsx
File size: 1805.86 KB
EXCEL REPORT TEST: PASS


In [112]:
# Run Streamlit application

import subprocess
import webbrowser
import time

print("STARTING NASSAU CANDY DASHBOARD")

print("Application:", APP_FILE)

streamlit_command = [
    sys.executable,
    "-m",
    "streamlit",
    "run",
    str(APP_FILE),
    "--server.headless",
    "true"
]

streamlit_process = subprocess.Popen(
    streamlit_command,
    cwd=str(APP_ROOT)
)

print("\nStreamlit process started.")
print("Open this address in your browser:")
print("http://localhost:8501")

print("\nKeep this notebook kernel running while using the dashboard.")

STARTING NASSAU CANDY DASHBOARD
Application: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\app.py

Streamlit process started.
Open this address in your browser:
http://localhost:8501

Keep this notebook kernel running while using the dashboard.


# Final output inventory

In [121]:
print("FINAL APPLICATION OUTPUT INVENTORY")

output_files = [
    APP_FILE,
    APP_CONFIG_PATH,
    recommendation_test_path,
    excel_report_path
]

final_inventory = []

for file in output_files:
    final_inventory.append({
        "File": str(file),
        "Exists": file.exists(),
        "Size (KB)": (
            round(file.stat().st_size / 1024, 2)
            if file.exists() else None
        )
    })

final_inventory_df = pd.DataFrame(final_inventory)

display(final_inventory_df)

print("\nAPPLICATION BUILD OUTPUT CHECK COMPLETED")

FINAL APPLICATION OUTPUT INVENTORY


,File,Exists,Size (KB)
0,C:\Users\hp\Nassau_Candy_Final_Submission\data...,True,14.56
1,C:\Users\hp\Nassau_Candy_Final_Submission\data...,True,0.95
2,C:\Users\hp\Nassau_Candy_Final_Submission\data...,True,0.59
3,C:\Users\hp\Nassau_Candy_Final_Submission\data...,True,1805.86



APPLICATION BUILD OUTPUT CHECK COMPLETED


In [127]:
import subprocess
import sys
import time

print("STARTING NASSAU CANDY STREAMLIT DASHBOARD")

streamlit_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        str(APP_FILE),
        "--server.headless=true",
        "--server.port=8501"
    ],
    cwd=str(APP_ROOT)
)

time.sleep(5)

if streamlit_process.poll() is None:
    print("Streamlit process is running.")
    print("Open: http://localhost:8501")
else:
    print("Streamlit stopped unexpectedly.")
    print("Exit code:", streamlit_process.returncode)

STARTING NASSAU CANDY STREAMLIT DASHBOARD
Streamlit stopped unexpectedly.
Exit code: 1


In [129]:
import subprocess
import sys

print("DIAGNOSING STREAMLIT STARTUP")

command = [
    sys.executable,
    "-m",
    "streamlit",
    "run",
    str(APP_FILE),
    "--server.headless=true",
    "--server.port=8501"
]

try:
    result = subprocess.run(
        command,
        cwd=str(APP_ROOT),
        capture_output=True,
        text=True,
        timeout=25
    )

    print("\nRETURN CODE:", result.returncode)

    print("\n--- STANDARD OUTPUT ---")
    print(result.stdout[-12000:] if result.stdout else "No standard output")

    print("\n--- ERROR OUTPUT ---")
    print(result.stderr[-12000:] if result.stderr else "No error output")

except subprocess.TimeoutExpired as error:
    print("Streamlit did not exit within 25 seconds.")
    print("This may indicate that the server started successfully.")
    print("Output:", error.stdout)
    print("Errors:", error.stderr)

except Exception as error:
    print("DIAGNOSTIC ERROR:", repr(error))

DIAGNOSING STREAMLIT STARTUP

RETURN CODE: 1

--- STANDARD OUTPUT ---
No standard output

--- ERROR OUTPUT ---
2026-11-01 19:28:34.319 Port 8501 is already in use



In [131]:
print("ENABLING EXCEL DOWNLOAD IN STREAMLIT APP")

app_text = APP_FILE.read_text(encoding="utf-8")

old_code = '''st.download_button(
    "Download product analysis Excel",
    data=(
        filtered.groupby("Product Name", as_index=False)
        .agg(
            Sales=("Sales", "sum"),
            Gross_Profit=("Gross Profit", "sum"),
            Units=("Units", "sum")
        )
        .to_excel if False else b""
    ),
    file_name="nassau_product_analysis.xlsx",
    disabled=True
)'''

new_code = '''from io import BytesIO

excel_buffer = BytesIO()

with pd.ExcelWriter(excel_buffer, engine="openpyxl") as writer:
    filtered.to_excel(
        writer,
        sheet_name="Filtered Transactions",
        index=False
    )

    filtered.groupby("Product Name", as_index=False).agg(
        Sales=("Sales", "sum"),
        Gross_Profit=("Gross Profit", "sum"),
        Units=("Units", "sum")
    ).to_excel(
        writer,
        sheet_name="Product Analysis",
        index=False
    )

    filtered.groupby("Region", as_index=False).agg(
        Sales=("Sales", "sum"),
        Gross_Profit=("Gross Profit", "sum")
    ).to_excel(
        writer,
        sheet_name="Region Analysis",
        index=False
    )

st.download_button(
    "Download analysis Excel",
    data=excel_buffer.getvalue(),
    file_name="nassau_dashboard_analysis.xlsx",
    mime="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
)'''

if old_code not in app_text:
    raise ValueError(
        "The expected Excel download section was not found. "
        "No changes were made."
    )

app_text = app_text.replace(old_code, new_code, 1)

APP_FILE.write_text(app_text, encoding="utf-8")

print("EXCEL DOWNLOAD ENABLED")

ENABLING EXCEL DOWNLOAD IN STREAMLIT APP
EXCEL DOWNLOAD ENABLED


In [133]:
import py_compile

py_compile.compile(
    str(APP_FILE),
    doraise=True
)

print("APPLICATION SYNTAX: PASS")
print("EXCEL DOWNLOAD SECTION UPDATED")

APPLICATION SYNTAX: PASS
EXCEL DOWNLOAD SECTION UPDATED


In [135]:
# Run Streamlit application

import subprocess
import webbrowser
import time

print("STARTING NASSAU CANDY DASHBOARD")

print("Application:", APP_FILE)

streamlit_command = [
    sys.executable,
    "-m",
    "streamlit",
    "run",
    str(APP_FILE),
    "--server.headless",
    "true"
]

streamlit_process = subprocess.Popen(
    streamlit_command,
    cwd=str(APP_ROOT)
)

print("\nStreamlit process started.")
print("Open this address in your browser:")
print("http://localhost:8501")

print("\nKeep this notebook kernel running while using the dashboard.")

STARTING NASSAU CANDY DASHBOARD
Application: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\app.py

Streamlit process started.
Open this address in your browser:
http://localhost:8501

Keep this notebook kernel running while using the dashboard.


In [137]:
print("CHECKING EXCEL DOWNLOAD SECTION")

app_lines = APP_FILE.read_text(encoding="utf-8").splitlines()

for i, line in enumerate(app_lines):
    if "Excel" in line or "excel" in line or "BytesIO" in line:
        start = max(0, i - 5)
        end = min(len(app_lines), i + 18)

        print(f"\n--- Lines {start + 1} to {end} ---")

        for j in range(start, end):
            print(f"{j + 1}: {app_lines[j]}")

print("\nINSPECTION COMPLETED")

CHECKING EXCEL DOWNLOAD SECTION

--- Lines 527 to 549 ---
527:     filtered.to_csv(index=False).encode("utf-8"),
528:     "nassau_filtered_analysis.csv",
529:     "text/csv"
530: )
531: 
532: from io import BytesIO
533: 
534: excel_buffer = BytesIO()
535: 
536: with pd.ExcelWriter(excel_buffer, engine="openpyxl") as writer:
537:     filtered.to_excel(
538:         writer,
539:         sheet_name="Filtered Transactions",
540:         index=False
541:     )
542: 
543:     filtered.groupby("Product Name", as_index=False).agg(
544:         Sales=("Sales", "sum"),
545:         Gross_Profit=("Gross Profit", "sum"),
546:         Units=("Units", "sum")
547:     ).to_excel(
548:         writer,
549:         sheet_name="Product Analysis",

--- Lines 529 to 551 ---
529:     "text/csv"
530: )
531: 
532: from io import BytesIO
533: 
534: excel_buffer = BytesIO()
535: 
536: with pd.ExcelWriter(excel_buffer, engine="openpyxl") as writer:
537:     filtered.to_excel(
538:         writer,
539:         s

In [139]:
import py_compile
import subprocess
import sys
import time

print("VALIDATING UPDATED DASHBOARD")

py_compile.compile(
    str(APP_FILE),
    doraise=True
)

print("Syntax check: PASS")

print("\nRESTARTING STREAMLIT")

try:
    if (
        "streamlit_process" in globals()
        and streamlit_process.poll() is None
    ):
        streamlit_process.terminate()
        streamlit_process.wait(timeout=10)
        print("Previous dashboard process stopped.")
except Exception as error:
    print("Previous process status:", error)

streamlit_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        str(APP_FILE),
        "--server.headless=true",
        "--server.port=8502"
    ],
    cwd=str(APP_ROOT)
)

time.sleep(5)

if streamlit_process.poll() is None:
    print("\nDASHBOARD RESTARTED")
    print("Open: http://localhost:8502")
else:
    print("\nSTARTUP FAILED")
    print("Exit code:", streamlit_process.returncode)

VALIDATING UPDATED DASHBOARD
Syntax check: PASS

RESTARTING STREAMLIT
Previous dashboard process stopped.

DASHBOARD RESTARTED
Open: http://localhost:8502


In [155]:
from pathlib import Path
import shutil
import py_compile

app_path = Path(APP_FILE)
backup_path = app_path.with_name("app_before_excel_sidebar.py")

if not backup_path.exists():
    raise FileNotFoundError(
        f"Recovery backup not found: {backup_path}"
    )

shutil.copy2(backup_path, app_path)

py_compile.compile(str(app_path), doraise=True)

print("RECOVERY SUCCESSFUL")
print("Original dashboard code restored.")
print("Syntax check: PASS")
print("Restored file:", app_path)

RECOVERY SUCCESSFUL
Original dashboard code restored.
Syntax check: PASS
Restored file: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\app.py


In [157]:
import py_compile
from pathlib import Path

app_path = Path(APP_FILE)
source = app_path.read_text(encoding="utf-8")

start_marker = "from io import BytesIO"
end_marker = '''st.caption(
    "Analysis is based on the selected records. "
    "Recommendations are indicative and not proven optimal."
)'''

start = source.rfind(start_marker)
end = source.find(end_marker, start)

if start == -1 or end == -1:
    raise ValueError("Excel export section not found. No changes made.")

replacement = '''from io import BytesIO

try:
    excel_buffer = BytesIO()

    with pd.ExcelWriter(excel_buffer, engine="openpyxl") as writer:
        filtered.to_excel(
            writer,
            sheet_name="Filtered Transactions",
            index=False
        )

        filtered.groupby("Product Name", as_index=False).agg(
            Sales=("Sales", "sum"),
            Gross_Profit=("Gross Profit", "sum"),
            Units=("Units", "sum")
        ).to_excel(
            writer,
            sheet_name="Product Analysis",
            index=False
        )

        filtered.groupby("Region", as_index=False).agg(
            Sales=("Sales", "sum"),
            Gross_Profit=("Gross Profit", "sum")
        ).to_excel(
            writer,
            sheet_name="Region Analysis",
            index=False
        )

    st.download_button(
        "Download analysis Excel",
        data=excel_buffer.getvalue(),
        file_name="nassau_dashboard_analysis.xlsx",
        mime="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        key="analysis_excel_download"
    )

except Exception as error:
    st.error(f"Excel export could not be generated: {error}")

'''

updated = source[:start] + replacement + source[end:]

app_path.write_text(updated, encoding="utf-8")

try:
    py_compile.compile(str(app_path), doraise=True)
    print("SUCCESS: Excel export updated.")
    print("CSV code left unchanged.")
    print("Home page code left unchanged.")
except Exception:
    app_path.write_text(source, encoding="utf-8")
    raise

SUCCESS: Excel export updated.
CSV code left unchanged.
Home page code left unchanged.


In [159]:
from pathlib import Path
import py_compile

app_path = Path(APP_FILE)
source = app_path.read_text(encoding="utf-8")

cache_function = '''

@st.cache_data(show_spinner=False)
def create_excel_report(data):
    from io import BytesIO

    buffer = BytesIO()

    with pd.ExcelWriter(buffer, engine="openpyxl") as writer:
        data.to_excel(
            writer,
            sheet_name="Filtered Transactions",
            index=False
        )

        data.groupby("Product Name", as_index=False).agg(
            Sales=("Sales", "sum"),
            Gross_Profit=("Gross Profit", "sum"),
            Units=("Units", "sum")
        ).to_excel(
            writer,
            sheet_name="Product Analysis",
            index=False
        )

        data.groupby("Region", as_index=False).agg(
            Sales=("Sales", "sum"),
            Gross_Profit=("Gross Profit", "sum")
        ).to_excel(
            writer,
            sheet_name="Region Analysis",
            index=False
        )

    return buffer.getvalue()

'''

if "def create_excel_report(data):" not in source:
    insertion = "def money(value):"
    if source.count(insertion) != 1:
        raise ValueError("Could not safely locate function insertion point.")

    source = source.replace(
        insertion,
        cache_function + "\n" + insertion,
        1
    )

old_start = source.rfind("from io import BytesIO")
old_end_marker = '''st.caption(
    "Analysis is based on the selected records. "
    "Recommendations are indicative and not proven optimal."
)'''

old_end = source.find(old_end_marker, old_start)

if old_start == -1 or old_end == -1:
    raise ValueError("Excel section not found. No changes saved.")

replacement = '''try:
    excel_data = create_excel_report(filtered)

    st.download_button(
        "Download analysis Excel",
        data=excel_data,
        file_name="nassau_dashboard_analysis.xlsx",
        mime="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        key="analysis_excel_download"
    )

except Exception as error:
    st.error(f"Excel export could not be generated: {error}")

'''

updated = source[:old_start] + replacement + source[old_end:]

backup = app_path.with_name("app_before_excel_cache.py")
backup.write_text(app_path.read_text(encoding="utf-8"), encoding="utf-8")

app_path.write_text(updated, encoding="utf-8")

try:
    py_compile.compile(str(app_path), doraise=True)
    print("EXCEL CACHE: ADDED")
    print("SYNTAX CHECK: PASS")
    print("BACKUP:", backup)
except Exception:
    app_path.write_text(backup.read_text(encoding="utf-8"), encoding="utf-8")
    raise

EXCEL CACHE: ADDED
SYNTAX CHECK: PASS
BACKUP: C:\Users\hp\Nassau_Candy_Final_Submission\data\processed\app_bundle\app_before_excel_cache.py


In [161]:
from pathlib import Path
import json
import py_compile
import pandas as pd

app_path = Path(APP_FILE)
source = app_path.read_text(encoding="utf-8")

print("=" * 55)
print("NASSAU CANDY DASHBOARD — FINAL VALIDATION")
print("=" * 55)

checks = {}

# Application syntax
try:
    py_compile.compile(str(app_path), doraise=True)
    checks["Application syntax"] = True
except Exception as error:
    checks["Application syntax"] = False
    print("Syntax error:", error)

# Configuration
try:
    with open(APP_ROOT / "app_config.json", encoding="utf-8") as f:
        config = json.load(f)

    data_path = Path(config["data_path"])
    checks["Application configuration"] = True
    checks["Processed dataset exists"] = data_path.exists()
except Exception as error:
    checks["Application configuration"] = False
    checks["Processed dataset exists"] = False
    print("Configuration error:", error)

# Dashboard pages
required_pages = [
    "Home",
    "Product Performance",
    "Region Analysis",
    "Ship Mode Analysis",
    "Factory Explorer",
    "Scenario Simulator",
    "Recommendations",
    "Risk and Impact"
]

for page_name in required_pages:
    checks[f"Page: {page_name}"] = page_name in source

# Interactive filters
checks["Product filter"] = 'st.sidebar.multiselect(' in source
checks["Region filter"] = source.count('st.sidebar.multiselect(') >= 2
checks["Ship mode filter"] = source.count('st.sidebar.multiselect(') >= 3

# Downloads
checks["CSV download"] = "st.download_button(" in source
checks["Excel download"] = "Download analysis Excel" in source

# Scenario and recommendation features
checks["Scenario controls"] = "Scenario Simulator" in source
checks["Recommendation page"] = "Recommendations" in source
checks["Risk analysis page"] = "Risk and Impact" in source

# Excel caching
checks["Excel report caching"] = "def create_excel_report(data):" in source

print("\nVALIDATION RESULTS")
print("-" * 55)

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} | {name}")

passed_count = sum(checks.values())
total_count = len(checks)

print("-" * 55)
print(f"TOTAL: {passed_count}/{total_count} checks passed")

if passed_count == total_count:
    print("\nALL STATIC VALIDATION CHECKS PASSED")
else:
    print("\nSOME CHECKS NEED ATTENTION")

NASSAU CANDY DASHBOARD — FINAL VALIDATION

VALIDATION RESULTS
-------------------------------------------------------
PASS | Application syntax
PASS | Application configuration
PASS | Processed dataset exists
PASS | Page: Home
PASS | Page: Product Performance
PASS | Page: Region Analysis
PASS | Page: Ship Mode Analysis
PASS | Page: Factory Explorer
PASS | Page: Scenario Simulator
PASS | Page: Recommendations
PASS | Page: Risk and Impact
PASS | Product filter
PASS | Region filter
PASS | Ship mode filter
PASS | CSV download
PASS | Excel download
PASS | Scenario controls
PASS | Recommendation page
PASS | Risk analysis page
PASS | Excel report caching
-------------------------------------------------------
TOTAL: 20/20 checks passed

ALL STATIC VALIDATION CHECKS PASSED
